# Random Forest

## Introduction

This notebook extends the modeling workflow for predicting customer churn by introducing a random forest model as an ensemble-based improvement over a single decision tree.

In the previous stage, a decision tree was used to capture non-linear relationships and interactions directly from the data. While decision trees are flexible and easy to interpret, they are also prone to overfitting, as small changes in the data can lead to significantly different tree structures and predictions.

A random forest addresses this limitation by combining multiple decision trees into an ensemble model. Instead of relying on a single tree, the algorithm trains many trees on different bootstrap samples of the data and introduces additional randomness by selecting a subset of features at each split. The final prediction is obtained by aggregating the outputs of all individual trees, typically through averaging probabilities in classification tasks.

This ensemble approach reduces variance and improves generalization performance, making random forests more robust and stable compared to a single decision tree. At the same time, the model retains the ability to capture complex non-linear relationships and interactions without requiring explicit feature engineering.

Unlike a single decision tree, which provides a clear and interpretable structure, a random forest trades off some interpretability for improved predictive performance. Instead of individual decision paths, the model is better understood through aggregate measures such as feature importance and overall predictive behavior.

The objective of this stage is to evaluate whether aggregating multiple trees leads to improved performance and more stable predictions, particularly in terms of generalization to unseen data. Special attention is given to the balance between model complexity and overfitting, as well as the impact of key hyperparameters such as the number of trees, maximum tree depth, and feature subsampling.

To ensure consistency across the modeling workflow, the same evaluation framework is retained. Model training and hyperparameter tuning are performed using cross-validation within the training data, while a separate validation set is used to guide model selection. A final hold-out test set is reserved exclusively for unbiased performance evaluation.

As in previous stages, the focus extends beyond classification accuracy to include the quality of predicted probabilities and the model’s ability to rank customers by churn risk. Evaluation metrics such as ROC-AUC, PR-AUC, and threshold-based measures are used to provide a comprehensive assessment of model performance.

The results of this notebook will demonstrate the benefits of ensemble learning and establish a more robust benchmark, forming a foundation for further exploration of advanced tree-based methods such as gradient boosting.

## Table of Contents

1. [Modeling Considerations](#modeling-considerations)
3. [Model Implementation](#model-implementation)
   - [Initial Grid Search](#initial-grid-definition-and-cross-validation)
   - [Refining Grid Search](#refined-hyperparameter-grid)
   - [Interpretability and Feature Importance](#interpretability-and-feature-importance)
4. [Test Set Evaluation](#test-set-evaluation)
5. [Executive Summary](#executive-summary)

## Modeling Considerations

Before fitting the random forest model, we outline several key considerations that influence model behavior and evaluation.

---

### Class Imbalance

The target variable exhibits moderate class imbalance, with approximately 26.5% of customers churning.

Random forests are generally more robust to class imbalance than single decision trees due to their ensemble nature. However, the model can still exhibit bias toward the majority class, particularly when aggregating predictions across many trees trained on similar distributions of the data.

As a result, accuracy is not used as a primary evaluation metric. Instead, we focus on recall, precision, F1 score, and PR-AUC, which provide a more informative assessment of performance on the minority class. Where appropriate, class weighting or threshold adjustment may be considered to further improve sensitivity to churners.

---

### Feature Scaling

Random forests do not require feature scaling. Like decision trees, splits are based on thresholding individual features rather than distance-based calculations, making the scale of the variables irrelevant to model behavior.

All features are therefore used in their original form, maintaining consistency with the data preparation pipeline and simplifying preprocessing.

---

### Feature Representation

Random forests retain the ability of decision trees to model non-linear relationships and interaction effects without requiring explicit feature engineering. Each tree in the ensemble learns its own set of hierarchical splits, and the aggregation of many such trees allows the model to capture complex patterns in the data.

Continuous variables are used directly, as the model learns optimal split thresholds internally. Manual discretization is unnecessary and may reduce predictive performance by discarding information.

Simplified representations of categorical or service-related features are also preserved, as they provide a compact feature space while still allowing the model to identify meaningful patterns through repeated splitting across trees.

---

### Model Complexity and Overfitting

While individual decision trees are prone to overfitting, random forests mitigate this risk through averaging across many trees trained on different bootstrap samples and feature subsets. This reduces variance and leads to more stable predictions.

However, model complexity still needs to be controlled. Key hyperparameters such as the number of trees, maximum depth, minimum samples per leaf, and the number of features considered at each split influence both performance and computational cost.

Increasing the number of trees generally improves performance but with diminishing returns and higher computational overhead. Depth and leaf constraints act as regularization mechanisms, helping to prevent individual trees from fitting noise in the data.

---

### Validation Strategy and Iterative Model Refinement

Hyperparameter tuning and model refinement are performed iteratively, which introduces a risk of overfitting to evaluation data if not properly controlled. Although random forests are more stable than single trees, repeated tuning can still lead to optimistic bias.

To address this, the training data is further split into a dedicated training subset and a separate validation set. Cross-validation is applied within the training subset to evaluate candidate hyperparameter configurations, while the validation set is used to guide model selection.

This separation ensures that model selection decisions are not based on the same data used for model fitting. A final hold-out test set is reserved exclusively for unbiased performance evaluation and is not used during model tuning.

---

### Interpretability and Feature Importance

Random forests sacrifice the straightforward interpretability of a single decision tree in exchange for improved performance and stability. Individual decision paths are no longer easily interpretable, as predictions are based on the aggregation of many trees.

Instead, model interpretation relies on global measures such as feature importance, which quantify the contribution of each variable to the model’s predictive performance. These measures provide insight into which features are most influential in predicting churn, although they do not capture local decision logic.

---

### Summary

Compared to a single decision tree, random forests improve predictive performance and stability by reducing variance through ensemble learning. They retain the ability to model non-linear relationships and interactions while mitigating the risk of overfitting associated with individual trees.

The modeling approach therefore emphasizes appropriate evaluation metrics for imbalanced data, careful tuning of key hyperparameters, and a robust validation strategy. With these considerations in place, the random forest model provides a strong and reliable benchmark for churn prediction.

With these considerations in place, we proceed to defining and training the random forest model.


## Model Implementation

This section implements the random forest modeling workflow described above, translating the conceptual approach into a reusable and consistent pipeline.

Random forests extend decision trees by combining multiple trees into an ensemble, allowing the model to capture non-linear relationships and interaction effects while improving stability and generalization performance.

As in the previous stage, the model operates directly on the same feature representation used across the pipeline. Continuous variables are retained in their original form, enabling each tree in the ensemble to learn optimal split thresholds directly from the data without requiring manual discretization. This preserves the full informational content of the features and ensures consistency across modeling approaches.

Categorical variables are encoded using one-hot encoding to ensure compatibility with the modeling framework. While tree-based models are less sensitive to encoding choices than linear models, numerical input is still required. The chosen representation provides a consistent feature space and allows different trees within the ensemble to exploit category-specific patterns.

The training, validation, and test datasets are loaded and prepared using the shared data preparation pipeline. Since random forests, like decision trees, are insensitive to feature scaling, no additional transformations are required beyond the predefined feature engineering steps.

Model training is performed using the `RandomForestClassifier` from scikit-learn. Unlike a single decision tree, model behavior is influenced by both individual tree complexity and ensemble-level parameters. Key hyperparameters include the number of trees, maximum depth, minimum samples per leaf, and the number of features considered at each split.

To identify an appropriate model specification, hyperparameters are explored using cross-validation within the training subset. This provides a systematic approach to evaluating different configurations and understanding the trade-offs between model complexity, performance, and computational cost.

Because hyperparameter tuning is performed iteratively, cross-validation is used as an internal model comparison tool rather than the final selection criterion. Candidate models identified through cross-validation are subsequently evaluated on a separate validation set, which is used to select the final model configuration.

The implementation proceeds in the following steps:

1. Load training, validation, and test data
2. Apply the data preparation pipeline
3. Define target and predictor variables
4. Define the random forest model
5. Perform hyperparameter exploration using cross-validation on the training subset
6. Evaluate candidate models on the validation set and select the final configuration
7. Retrain the selected model on the combined training data (optional)
8. Evaluate final performance on the test set

We begin by loading the data.


In [1]:
import polars as pl
from sklearn.model_selection import train_test_split

# Load datasets
train_df = pl.read_csv("./data/processed/06_dpp_train_df.csv")
test_df = pl.read_csv("./data/processed/06_dpp_test_df.csv")

selected_cols = [
    'SeniorCitizenRelevel',
    'Partner',
    'Dependents',
    'tenure',
    'Contract',
    'PaperlessBilling',
    'MonthlyCharges',
    'Churn',
    'AdditionalInternetServicesCount',
    'StreamingServicesCount',
    'PaymentMethod_bin_3'
]

train_df = train_df.select(selected_cols)
test_df = test_df.select(selected_cols)

# Convert to pandas for sklearn
train_pd = train_df.to_pandas()

# Stratified split: 25% of current train -> validation
train_sub_pd, val_pd = train_test_split(
    train_pd,
    test_size=0.25,
    stratify=train_pd["Churn"],
    random_state=42
)

# Back to polars
train_sub_df = pl.from_pandas(train_sub_pd)
val_df = pl.from_pandas(val_pd)

# Shapes
print("Train subset shape:", train_sub_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Class proportions helper
def class_proportions(df, name):
    print(f"\n{name} class proportions:")
    print(
        df.group_by("Churn")
        .len()
        .with_columns(
            (pl.col("len") / pl.col("len").sum()).alias("proportion")
        )
        .sort("Churn")
    )

class_proportions(train_sub_df, "Train subset")
class_proportions(val_df, "Validation")
class_proportions(test_df, "Test")

Train subset shape: (4225, 11)
Validation shape: (1409, 11)
Test shape: (1409, 11)

Train subset class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 3104 ┆ 0.734675   │
│ Yes   ┆ 1121 ┆ 0.265325   │
└───────┴──────┴────────────┘

Validation class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘

Test class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘


The datasets are now loaded and partitioned into training, validation, and test sets. We proceed by preparing the data, ensuring consistent encoding and feature representation across all splits.

In [ ]:
from src.utils.data_preparation_models import prepare_tree_features

categorical_cols = ['SeniorCitizenRelevel',
                    'Partner',
                    'Dependents',
                    'Contract',
                    'PaperlessBilling',
                    'PaymentMethod_bin_3']

target_col = "Churn"

numerical_cols = ['tenure', 
                  'MonthlyCharges', 
                  'AdditionalInternetServicesCount', 
                  'StreamingServicesCount']   

categorical_orders = { "SeniorCitizenRelevel": ["No", "Yes"], 
                      "Partner": ["No", "Yes"], 
                      "Dependents": ["No", "Yes"], 
                      "PaperlessBilling": ["No", "Yes"], 
                      "Contract": ["Month-to-month", "One year", "Two year"] }

train_X, train_y = prepare_tree_features(
    df=train_sub_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col
)

validation_X, validation_y = prepare_tree_features(
    df=val_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col,
    reference_columns=train_X.columns.tolist()
)

print(train_X.shape)
print(validation_X.shape)

(4225, 18)
(1409, 18)


Categorical variables are one-hot encoded to provide a fully numeric input matrix for the model.

Some features created during feature engineering are represented as short-range integer variables. These are retained as ordinal numeric features rather than being one-hot encoded. Preserving their ordering allows the decision tree to learn threshold-based splits directly, which is a core aspect of its learning mechanism.

The resulting design matrix therefore consists of a combination of one-hot encoded categorical variables, ordinal integer features, and continuous variables, providing a flexible feature space capable of capturing both discrete and continuous patterns in the data.

### Initial Grid Definition and Cross-Validation

The random forest model is initially explored using cross-validated grid search to identify promising regions of the hyperparameter space. This approach evaluates multiple configurations across different data splits within the training subset, providing a stable estimate of relative model performance and reducing sensitivity to a single data partition.

Model comparison during this stage is based on PR-AUC (average precision), which is well-suited for imbalanced classification problems and emphasizes the model’s ability to correctly rank and identify churners.

The hyperparameter grid is designed to explore both individual tree complexity and ensemble-level behavior. Key parameters include the number of trees, maximum depth, minimum samples per split, minimum samples per leaf, and the number of features considered at each split. Together, these parameters control model flexibility, variance reduction, and computational cost.

Unlike a single decision tree, where complexity is primarily governed by structural constraints, random forests introduce additional randomness through bootstrap sampling and feature subsampling. These mechanisms help reduce overfitting but also introduce new dimensions to the tuning process, particularly in balancing bias, variance, and runtime efficiency.

At this stage, cross-validation is used as an internal model comparison tool rather than the final selection criterion. Candidate configurations identified through cross-validation will be further evaluated on a separate validation set to guide final model selection.

The tuning process therefore balances model capacity, stability, and computational efficiency, aiming to identify configurations that generalize well while remaining robust across validation folds.


In [4]:
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [4, 6, 8, 10],
    "min_samples_leaf": [1, 5, 10],
    "min_samples_split": [2, 10],
    "max_features": ["sqrt", 0.3, 0.5],
    "criterion": ["gini", "entropy"],
    "bootstrap": [True],
    "class_weight": [None, "balanced"]
}

### Hyperparameter Grid Design

The hyperparameter grid is designed to explore both ensemble-level behavior and individual tree complexity, while maintaining a balance between model performance and computational efficiency.

---

#### Number of Trees (`n_estimators`)

The number of trees in the forest controls the strength and stability of the ensemble.

A range of 200 to 500 trees is considered. This provides sufficient model capacity to reduce variance and stabilize predictions, while avoiding excessive computational cost. Increasing the number of trees beyond this range typically yields diminishing returns in performance relative to the additional training time.

---

#### Tree Depth (`max_depth`)

The maximum depth parameter controls the complexity of individual trees within the ensemble.

Values between 4 and 10 are explored to limit excessive tree growth while still allowing the model to capture non-linear relationships and interactions. Compared to a single decision tree, random forests rely less on deep trees, as ensemble averaging already mitigates overfitting.

---

#### Minimum Samples per Leaf (`min_samples_leaf`)

This parameter defines the minimum number of observations required in each terminal node.

Values of 1, 5, and 10 are included to control the smoothness of predictions. Larger values enforce more conservative splits, reducing sensitivity to noise and improving generalization, particularly in the presence of class imbalance.

---

#### Minimum Samples for Split (`min_samples_split`)

This parameter determines the minimum number of observations required to consider a split.

Values of 2 and 10 are explored to control how aggressively trees grow. Higher values act as a regularization mechanism, preventing the model from learning overly specific patterns in the training data.

---

#### Feature Subsampling (`max_features`)

The `max_features` parameter controls the number of predictors randomly considered at each split and is a key component of the random forest algorithm.

Both standard heuristics (`"sqrt"`) and fraction-based values (0.3 and 0.5) are included. Smaller subsets increase randomness and reduce correlation between trees, improving generalization, while larger subsets allow individual trees to make stronger splits at the cost of increased correlation.

---

#### Split Criterion (`criterion`)

The split criterion determines how node impurity is measured.

Both Gini impurity and entropy are evaluated. While these criteria often yield similar performance, including both allows for empirical comparison and ensures that the chosen model is not sensitive to this specification.

---

#### Bootstrap Sampling (`bootstrap`)

Bootstrap sampling is enabled for all configurations, as it is a fundamental component of the random forest algorithm.

Each tree is trained on a different bootstrap sample of the data, which introduces variability across trees and contributes to variance reduction through ensemble averaging.

---

#### Class Weights (`class_weight`)

To account for class imbalance, both unweighted and balanced class weighting schemes are evaluated.

The `"balanced"` option adjusts class weights inversely proportional to class frequencies, increasing the importance of the minority class during training. This can improve the model’s ability to detect churners, particularly when optimizing metrics such as PR-AUC.

---

### Summary

The grid is designed to jointly explore model capacity, regularization, and ensemble diversity. Tree-level parameters control the flexibility of individual learners, while ensemble-level parameters govern stability and variance reduction.

This structured exploration enables identification of configurations that achieve strong predictive performance while remaining robust and computationally efficient.


Run cross validation on grid search

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

tree = RandomForestClassifier(random_state=42,
                              n_jobs=1)

grid = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid.fit(train_X, train_y)

Fitting 5 folds for each of 576 candidates, totalling 2880 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True], 'class_weight': [None, 'balanced'], 'criterion': ['gini', 'entropy'], 'max_depth': [4, 6, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold

Extract the best candidates to validate

In [7]:
import pandas as pd

cv_results = pd.DataFrame(grid.cv_results_)

cv_summary = (
    cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "params"
        ]
    ]
    .sort_values(
        ["rank_test_score", "mean_test_score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

cv_summary.head(10)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.659634,0.052782,0.726578,0.011307,"{'bootstrap': True, 'class_weight': None, 'cri..."
1,1,0.659634,0.052782,0.726578,0.011307,"{'bootstrap': True, 'class_weight': None, 'cri..."
2,3,0.659402,0.053408,0.729843,0.011290,"{'bootstrap': True, 'class_weight': None, 'cri..."
3,4,0.659342,0.053231,0.721039,0.011459,"{'bootstrap': True, 'class_weight': None, 'cri..."
4,4,0.659342,0.053231,0.721039,0.011459,"{'bootstrap': True, 'class_weight': None, 'cri..."
5,6,0.659329,0.052922,0.732074,0.010927,"{'bootstrap': True, 'class_weight': None, 'cri..."
6,7,0.659292,0.053531,0.732386,0.010274,"{'bootstrap': True, 'class_weight': None, 'cri..."
7,8,0.659289,0.053054,0.729767,0.011507,"{'bootstrap': True, 'class_weight': None, 'cri..."
8,9,0.659231,0.053525,0.733766,0.010801,"{'bootstrap': True, 'class_weight': None, 'cri..."
9,9,0.659231,0.053525,0.733766,0.010801,"{'bootstrap': True, 'class_weight': None, 'cri..."


With Random Forest we will inspect 10 best candidates.

In [9]:
top_candidates = cv_summary.head(10).copy()

Run validation on best selected candidates.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from src.utils.classification_metrics import compute_classification_metrics

import pandas as pd

def evaluate_rf_candidates_on_validation(
    top_candidates,
    train_X,
    train_y,
    validation_X,
    validation_y,
    random_state=42,
    sort_by="pr_auc"
):
    """
    Fit Random Forest candidate models on training data and evaluate on validation set.

    Parameters
    ----------
    top_candidates : pd.DataFrame
        DataFrame containing at least:
        - 'params'
        - 'rank_test_score'
        - 'mean_test_score'
        - 'std_test_score'

    train_X, train_y : training data
    validation_X, validation_y : validation data

    random_state : int
    sort_by : metric to sort results by (default 'pr_auc')

    Returns
    -------
    pd.DataFrame
        Validation results sorted by chosen metric
    """

    candidate_results = []

    for _, row in top_candidates.iterrows():
        params = row["params"]

        model = RandomForestClassifier(
            **params,
            random_state=random_state,
            n_jobs=1   # avoid nested parallelism
        )

        model.fit(train_X, train_y)

        val_prob = model.predict_proba(validation_X)[:, 1]
        val_pred = model.predict(validation_X)

        metrics = compute_classification_metrics(
            y_true=validation_y,
            y_pred=val_pred,
            y_prob=val_prob
        )

        candidate_results.append({
            "cv_rank": row["rank_test_score"],
            "cv_pr_auc": row["mean_test_score"],
            "cv_std": row["std_test_score"],
            "params": params,
            **metrics
        })

    results_df = (
        pd.DataFrame(candidate_results)
        .sort_values(by=sort_by, ascending=False)
        .reset_index(drop=True)
    )

    return results_df

In [11]:
candidate_results_df = evaluate_rf_candidates_on_validation(
    top_candidates=top_candidates,
    train_X=train_X,
    train_y=train_y,
    validation_X=validation_X,
    validation_y=validation_y
)

candidate_results_df

,cv_rank,cv_pr_auc,cv_std,params,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,8,0.659289,0.053054,"{'bootstrap': True, 'class_weight': None, 'cri...",0.784244,0.628676,0.457219,0.529412,0.815982,0.610133,171,934,101,203
1,4,0.659342,0.053231,"{'bootstrap': True, 'class_weight': None, 'cri...",0.787083,0.638060,0.457219,0.532710,0.818462,0.610071,171,938,97,203
2,4,0.659342,0.053231,"{'bootstrap': True, 'class_weight': None, 'cri...",0.787083,0.638060,0.457219,0.532710,0.818462,0.610071,171,938,97,203
3,7,0.659292,0.053531,"{'bootstrap': True, 'class_weight': None, 'cri...",0.784954,0.631970,0.454545,0.528771,0.816583,0.608756,170,936,99,204
4,9,0.659231,0.053525,"{'bootstrap': True, 'class_weight': None, 'cri...",0.784954,0.631970,0.454545,0.528771,0.816809,0.608623,170,936,99,204
5,9,0.659231,0.053525,"{'bootstrap': True, 'class_weight': None, 'cri...",0.784954,0.631970,0.454545,0.528771,0.816809,0.608623,170,936,99,204
6,3,0.659402,0.053408,"{'bootstrap': True, 'class_weight': None, 'cri...",0.784244,0.631579,0.449198,0.525000,0.816103,0.608342,168,937,98,206
7,1,0.659634,0.052782,"{'bootstrap': True, 'class_weight': None, 'cri...",0.785664,0.635338,0.451872,0.528125,0.816571,0.608274,169,938,97,205
8,1,0.659634,0.052782,"{'bootstrap': True, 'class_weight': None, 'cri...",0.785664,0.635338,0.451872,0.528125,0.816571,0.608274,169,938,97,205
9,6,0.659329,0.052922,"{'bootstrap': True, 'class_weight': None, 'cri...",0.783534,0.630189,0.446524,0.522692,0.816256,0.607401,167,937,98,207


### First Model Selection

We are selecting model with index 1 as the first candidate (second row).

The selection is based on validation set performance rather than cross-validation ranking alone.

While several candidate configurations achieved nearly identical cross-validated PR-AUC, small differences in validation performance reveal meaningful distinctions in classification behavior.

In particular, the selected model achieves a slightly lower PR-AUC compared to the top-ranked candidate, but provides a noticeable improvement in F1 score and precision while maintaining the same level of recall. This indicates a better balance between identifying churners and limiting false positives.

Given the negligible difference in PR-AUC and the improved classification performance, this configuration is chosen as the first selected model.

This highlights the importance of evaluating multiple candidate models on a separate validation set, as cross-validation alone may not capture subtle but practically relevant differences in model behavior.

In [14]:
model_1_results = candidate_results_df.loc[1]
print("Model results:")
print(model_1_results)
model_1_params = model_1_results["params"]
print("Model Parameters:")
print(model_1_params)


Model results:
cv_rank                                                      4
cv_pr_auc                                             0.659342
cv_std                                                0.053231
params       {'bootstrap': True, 'class_weight': None, 'cri...
accuracy                                              0.787083
precision                                              0.63806
recall                                                0.457219
f1                                                     0.53271
auc                                                   0.818462
pr_auc                                                0.610071
tp                                                         171
tn                                                         938
fp                                                          97
fn                                                         203
Name: 1, dtype: object
Model Parameters:
{'bootstrap': True, 'class_weight': None, 'criterion': 'entrop

### Refined Hyperparameter Grid

Based on the initial validation results, a refined hyperparameter grid is defined around the strongest-performing random forest configuration. Rather than repeating a broad search, the refinement step focuses on a narrower region of the hyperparameter space centered on the previously selected candidate model.

Parameters that showed clear preference in the initial search, such as the split criterion and feature subsampling ratio, are fixed to reduce computational cost and concentrate the search on the most relevant dimensions. The refinement therefore focuses primarily on the number of trees, tree depth, and node-size regularization parameters.

This targeted search aims to determine whether small adjustments around the initially selected configuration lead to further improvements in validation performance while maintaining model stability.

In [15]:
param_grid_refined = {
    "n_estimators": [500, 750],
    "max_depth": [5, 6, 7],
    "min_samples_leaf": [7, 10, 12],
    "min_samples_split": [2, 5],
    "max_features": [0.5],
    "criterion": ["entropy"],
    "bootstrap": [True],
    "class_weight": [None, "balanced"]
}

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

tree = RandomForestClassifier(random_state=42,
                              n_jobs=1)

grid_refined = GridSearchCV(
    estimator=tree,
    param_grid=param_grid_refined,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_refined.fit(train_X, train_y)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True], 'class_weight': [None, 'balanced'], 'criterion': ['entropy'], 'max_depth': [5, 6, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and par

In [18]:
import pandas as pd

cv_refined_results = pd.DataFrame(grid_refined.cv_results_)

cv_refined_summary = (
    cv_refined_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "params"
        ]
    ]
    .sort_values(
        ["rank_test_score", "mean_test_score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

cv_refined_summary.head(10)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.660550,0.053993,0.739697,0.010020,"{'bootstrap': True, 'class_weight': None, 'cri..."
1,1,0.660550,0.053993,0.739697,0.010020,"{'bootstrap': True, 'class_weight': None, 'cri..."
2,3,0.660486,0.053719,0.744686,0.010109,"{'bootstrap': True, 'class_weight': None, 'cri..."
3,3,0.660486,0.053719,0.744686,0.010109,"{'bootstrap': True, 'class_weight': None, 'cri..."
4,5,0.660033,0.054200,0.753335,0.009631,"{'bootstrap': True, 'class_weight': None, 'cri..."
5,5,0.660033,0.054200,0.753335,0.009631,"{'bootstrap': True, 'class_weight': None, 'cri..."
6,7,0.659781,0.053683,0.739927,0.010121,"{'bootstrap': True, 'class_weight': None, 'cri..."
7,7,0.659781,0.053683,0.739927,0.010121,"{'bootstrap': True, 'class_weight': None, 'cri..."
8,9,0.659715,0.053358,0.748544,0.010517,"{'bootstrap': True, 'class_weight': 'balanced'..."
9,9,0.659715,0.053358,0.748544,0.010517,"{'bootstrap': True, 'class_weight': 'balanced'..."


In [20]:
top_candidates_refined = cv_refined_summary.head(10).copy()

In [21]:
candidate_refined_results_df = evaluate_rf_candidates_on_validation(
    top_candidates=top_candidates_refined,
    train_X=train_X,
    train_y=train_y,
    validation_X=validation_X,
    validation_y=validation_y
)

candidate_refined_results_df

,cv_rank,cv_pr_auc,cv_std,params,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,1,0.660550,0.053993,"{'bootstrap': True, 'class_weight': None, 'cri...",0.788502,0.637681,0.470588,0.541538,0.819547,0.613771,176,935,100,198
1,1,0.660550,0.053993,"{'bootstrap': True, 'class_weight': None, 'cri...",0.788502,0.637681,0.470588,0.541538,0.819547,0.613771,176,935,100,198
2,7,0.659781,0.053683,"{'bootstrap': True, 'class_weight': None, 'cri...",0.789922,0.641304,0.473262,0.544615,0.819445,0.612695,177,936,99,197
3,7,0.659781,0.053683,"{'bootstrap': True, 'class_weight': None, 'cri...",0.789922,0.641304,0.473262,0.544615,0.819445,0.612695,177,936,99,197
4,3,0.660486,0.053719,"{'bootstrap': True, 'class_weight': None, 'cri...",0.787083,0.634058,0.467914,0.538462,0.819592,0.612578,175,934,101,199
5,3,0.660486,0.053719,"{'bootstrap': True, 'class_weight': None, 'cri...",0.787083,0.634058,0.467914,0.538462,0.819592,0.612578,175,934,101,199
6,5,0.660033,0.054200,"{'bootstrap': True, 'class_weight': None, 'cri...",0.786373,0.633700,0.462567,0.534776,0.817967,0.611610,173,935,100,201
7,5,0.660033,0.054200,"{'bootstrap': True, 'class_weight': None, 'cri...",0.786373,0.633700,0.462567,0.534776,0.817967,0.611610,173,935,100,201
8,9,0.659715,0.053358,"{'bootstrap': True, 'class_weight': 'balanced'...",0.732434,0.497288,0.735294,0.593312,0.818228,0.610701,275,757,278,99
9,9,0.659715,0.053358,"{'bootstrap': True, 'class_weight': 'balanced'...",0.732434,0.497288,0.735294,0.593312,0.818228,0.610701,275,757,278,99


### Refined Model Selection

The refined hyperparameter search results indicate that model performance has reached a stable region, with only marginal differences between candidate configurations.

Several models achieve nearly identical cross-validated and validation performance, with differences that are smaller than the observed variability across cross-validation folds. This suggests that further improvements through hyperparameter tuning are limited.

Among the evaluated candidates, the configuration corresponding to index 2 (and equivalently index 3) is selected as the refined model. While its PR-AUC is slightly lower than the top-ranked configuration, it achieves consistent improvements in precision, recall, and F1 score. This indicates a more balanced classification performance without sacrificing ranking quality.

In contrast, configurations using balanced class weights significantly increase recall but at the cost of a substantial decrease in precision. Although this behavior may be desirable in certain applications, it does not improve overall ranking performance and introduces a higher number of false positives.

Given the minimal differences in PR-AUC and the consistent improvements in classification metrics, the selected configuration represents the best trade-off between performance, stability, and practical usefulness.

These results also highlight that the model has reached a performance plateau, where further hyperparameter refinement yields diminishing returns. The focus therefore shifts from tuning to final model evaluation and comparison.

In [22]:
final_results = candidate_results_df.loc[2]
print("Final model results:")
print(final_results)
final_model_params = final_results["params"]
print("Final model results:")
print(final_model_params)

Final model results:
cv_rank                                                      4
cv_pr_auc                                             0.659342
cv_std                                                0.053231
params       {'bootstrap': True, 'class_weight': None, 'cri...
accuracy                                              0.787083
precision                                              0.63806
recall                                                0.457219
f1                                                     0.53271
auc                                                   0.818462
pr_auc                                                0.610071
tp                                                         171
tn                                                         938
fp                                                          97
fn                                                         203
Name: 2, dtype: object
Final model results:
{'bootstrap': True, 'class_weight': None, 'criterion'

### Training best model on whole train set

In [25]:
train_val_X = pd.concat([train_X, validation_X], axis=0)
train_val_y = pd.concat([train_y, validation_y], axis=0)

from sklearn.ensemble import RandomForestClassifier

final_model = RandomForestClassifier(
    **final_model_params,
    random_state=42,
    n_jobs=-1
)

final_model.fit(train_val_X, train_val_y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

### Model Interpretability

Unlike a single decision tree, a random forest does not provide a straightforward, interpretable structure of decision rules. Predictions are based on the aggregation of many trees, making individual decision paths difficult to trace.

To assess model behavior, global feature importance is used. Feature importance measures the contribution of each variable to reducing impurity across all trees in the ensemble. This provides insight into which features are most influential in predicting churn.

### Feature Importance

In [27]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": train_X.columns,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
6,Contract_Month-to-month,0.324945
14,tenure,0.176834
15,MonthlyCharges,0.174817
8,Contract_Two year,0.104670
12,PaymentMethod_bin_3_Electronic check,0.057108
7,Contract_One year,0.040688
16,AdditionalInternetServicesCount,0.030506
17,StreamingServicesCount,0.027324
10,PaperlessBilling_Yes,0.014645
9,PaperlessBilling_No,0.012123


The feature importance analysis shows that model predictions are driven primarily by a small subset of variables.

Contract type is the dominant predictor, with month-to-month contracts contributing the most to churn prediction. This is followed by tenure and monthly charges, both of which capture customer lifecycle and pricing effects. Together, these variables account for the majority of the model’s predictive power.

A second group of features, including payment method and aggregated service usage indicators, provides additional but smaller contributions. These variables help refine predictions but do not fundamentally drive model behavior.

The remaining features have relatively low importance values. While they are occasionally used by individual trees within the ensemble, their overall contribution to model performance is limited. This is expected in random forests, where feature importance is distributed across many trees and correlated variables may share predictive signal.

Overall, the results indicate that churn is primarily explained by contract structure, customer tenure, and pricing, with other variables providing incremental improvements in predictive accuracy.

In [28]:
### Store the model for the consistency

import pickle

model_package = {
    # model
    "model": final_model,
    # data preparation setup
    "categorical_cols":categorical_cols,
    "numerical_cols":numerical_cols,
    "categorical_orders":categorical_orders,
    "target_col":target_col,
    "reference_columns": train_X.columns.tolist()
}

with open("./models/random_forest_final.pkl", "wb") as f:
    pickle.dump(model_package, f)

## Test Set Evaluation

After completing model selection and hyperparameter refinement, the final random forest model is evaluated on the hold-out test set.

The test set has not been used at any stage of model training, cross-validation, or validation-based model selection. As a result, it provides an unbiased estimate of the model’s generalization performance on unseen data.

The final model is first retrained on the combined training and validation datasets to ensure that all available information is utilized. This step improves parameter estimation while preserving the independence of the test set for evaluation.

Model performance is assessed using the same metrics applied throughout the modeling process, including ROC-AUC, PR-AUC, and threshold-based classification metrics such as precision, recall, and F1 score. This ensures consistency and enables direct comparison with earlier models.

The results reported in this section represent the final estimate of model performance and form the basis for comparing the random forest model with previously developed approaches.

In [29]:
from src.utils.classification_metrics import compute_classification_metrics

categorical_cols = ['SeniorCitizenRelevel',
                    'Partner',
                    'Dependents',
                    'Contract',
                    'PaperlessBilling',
                    'PaymentMethod_bin_3']

target_col = "Churn"

numerical_cols = ['tenure', 
                  'MonthlyCharges', 
                  'AdditionalInternetServicesCount', 
                  'StreamingServicesCount']   

categorical_orders = { "SeniorCitizenRelevel": ["No", "Yes"], 
                      "Partner": ["No", "Yes"], 
                      "Dependents": ["No", "Yes"], 
                      "PaperlessBilling": ["No", "Yes"], 
                      "Contract": ["Month-to-month", "One year", "Two year"] }

test_X_reduced, test_y_reduced = prepare_tree_features(
    df=test_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col,
    reference_columns=train_X.columns.tolist()
)

In [30]:
test_probs = final_model.predict_proba(test_X_reduced)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

metrics = compute_classification_metrics(
    y_true=test_y_reduced,
    y_pred=test_preds,
    y_prob=test_probs
)

In [31]:
metrics_df = (
    pd.DataFrame([metrics])
    .drop(columns=["tp", "tn", "fp", "fn"])
    .round(4)
)

metrics_df

,accuracy,precision,recall,f1,auc,pr_auc
0,0.7913,0.6481,0.4679,0.5435,0.8453,0.6775


In [32]:
conf_matrix = pd.DataFrame(
    [[metrics["tn"], metrics["fp"]],
     [metrics["fn"], metrics["tp"]]],
    index=["Actual 0", "Actual 1"],
    columns=["Pred 0", "Pred 1"]
)

conf_matrix

,Pred 0,Pred 1
Actual 0,940,95
Actual 1,199,175


he final random forest model demonstrates strong and consistent performance on the hold-out test set, indicating good generalization to unseen data.

The model achieves balanced classification performance, with an F1 score of 0.5435, reflecting a stable trade-off between precision (0.6481) and recall (0.4679). This suggests that the model is effective at identifying churners while maintaining a relatively low rate of false positives.

From a ranking perspective, the model achieves a ROC-AUC of 0.8453 and a PR-AUC of 0.6775, indicating strong ability to distinguish between churners and non-churners and to prioritize high-risk customers. These results are consistent with validation performance, suggesting that the model has not overfit during the tuning process.

Overall, the model provides reliable probability estimates and maintains stable performance across evaluation datasets. This supports its suitability for applications where customers need to be ranked by churn risk or targeted based on predicted likelihood of churn.

The consistency between cross-validation, validation, and test results further indicates that the modeling approach and validation strategy were effective in controlling overfitting and selecting a robust final configuration.

## Executive Summary

This section presents the final results of the random forest model developed for predicting customer churn.

The model builds on earlier stages of the workflow by leveraging an ensemble of decision trees to capture non-linear relationships and interaction effects while improving predictive stability. Hyperparameters were tuned using cross-validation, and candidate models were evaluated on a separate validation set to ensure robust model selection.

The final model was retrained on the combined training and validation data and evaluated on a hold-out test set to obtain an unbiased estimate of performance. The results indicate strong generalization, with consistent performance across cross-validation, validation, and test datasets.

On the test set, the model achieves balanced classification performance, with an F1 score of 0.5435, reflecting a stable trade-off between precision (0.6481) and recall (0.4679). In addition, the model demonstrates strong ranking capability, with a ROC-AUC of 0.8453 and a PR-AUC of 0.6775, indicating effective separation of churners from non-churners and reliable prioritization of high-risk customers.

Feature importance analysis shows that the model is primarily driven by contract type, customer tenure, and monthly charges, with additional contributions from payment method and service-related features. This confirms that the model captures meaningful patterns related to customer lifecycle and pricing behavior.

Overall, the random forest model provides a robust and well-calibrated predictive solution for churn estimation. Its ability to balance classification performance with strong ranking capability makes it suitable for practical applications such as targeted retention campaigns and customer risk scoring.